# 🧠 Building a Deep Q-Network with Keras

In this notebook I build a **Deep Q-Network (DQN)** from scratch with Keras and use it to solve the
`CartPole-v1` control problem: keep a pole balanced upright on a moving cart for as long as possible.

This is my first end-to-end reinforcement learning agent where *every* piece is hand-written — the
network, the replay buffer, the exploration policy, the Bellman update, and the training loop.

## 📋 Overview

Supervised learning gives me a dataset with labels. Reinforcement learning gives me none of that.
Instead I get an **environment** that reacts to what I do, and a scalar **reward** telling me whether
what I did was good. The agent has to generate its own training data by acting.

This feels close to control engineering: a controller observing a plant, applying an input, measuring
the response, and adjusting. The difference is that here I don't derive the control law analytically —
the network learns it from experience.

| What I build | Why it exists |
|---|---|
| 🌍 Gymnasium `CartPole-v1` environment | The plant I'm trying to control |
| 🏗️ Keras Q-network | Estimates the value of each action from a continuous state |
| 📥 Replay buffer | Decorrelates training samples so gradient descent behaves |
| 🎯 Epsilon-greedy policy | Balances trying new things vs. using what I know |
| 🔄 Bellman update | Turns reinforcement learning into a regression problem |
| ⚙️ Training loop | Where the agent actually interacts and learns |
| 📊 Evaluation loop | Measures the trained policy with exploration switched off |

**What CartPole actually is.** A cart slides along a frictionless track with a pole hinged on top.
The state is 4 continuous numbers:

$$s = [x, \dot{x}, \theta, \dot{\theta}]$$

Where:
- $x$ = cart position
- $\dot{x}$ = cart velocity
- $\theta$ = pole angle from vertical
- $\dot{\theta}$ = pole angular velocity

There are exactly 2 actions: push left, or push right. Every timestep the pole stays up earns
$+1$ reward. The episode ends when $|\theta|$ exceeds ~12° or the cart runs off the track.

## 🧩 Theory

### The reinforcement learning loop

At each timestep $t$ the agent observes state $s_t$, picks action $a_t$, and the environment returns
a reward $r_t$ and a next state $s_{t+1}$:

```
        action a_t
  ┌──────────────────────►┌─────────────┐
  │                       │ ENVIRONMENT │
┌─┴─────┐                 │  (CartPole) │
│ AGENT │                 └──────┬──────┘
└─▲─────┘                        │
  │   state s_{t+1}, reward r_t  │
  └──────────────────────────────┘
```

This is a closed feedback loop — structurally the same shape as an automatic gain control or a
power-control loop in a radio link. The agent is the controller, the reward is the error signal,
and the policy is the control law being tuned online.

### 🔢 Q-values and the Bellman equation

The **Q-value** $Q(s, a)$ answers one question: *if I'm in state $s$ and I take action $a$, how much
total reward should I expect from here to the end of the episode?*

The word "total" needs discounting, otherwise it can grow without bound and near-term reward gets
treated identically to reward 200 steps away:

$$Q(s_t, a_t) = \mathbb{E}\left[\sum_{k=0}^{\infty} \gamma^k \, r_{t+k}\right]$$

Where $\gamma \in [0, 1)$ is the **discount factor**. With $\gamma = 0.95$, a reward 20 steps in the
future is worth $0.95^{20} \approx 0.36$ of the same reward right now.

This is exactly an exponential forgetting factor — the same structure as the decay in an
exponentially-weighted moving average used to smooth a noisy RSSI measurement. Recent evidence
dominates; old evidence fades geometrically.

The **Bellman equation** is the recursion that makes this computable. The value of acting now is
the immediate reward plus the discounted value of acting optimally from wherever I land:

$$Q(s, a) = r + \gamma \max_{a'} Q(s', a')$$

### 🏗️ Why a neural network instead of a table

Classic Q-learning stores $Q$ in a table: one row per state, one column per action. That works when
states are discrete and countable. CartPole's state is **4 continuous real numbers** — there is no
finite table to fill in, and the agent will essentially never visit the exact same state twice.

So I replace the table with a function approximator:

$$Q(s,a) \;\approx\; Q_\theta(s,a)$$

A network with weights $\theta$ that takes a state vector and outputs one Q-value per action. It
**generalizes**: a state it has never seen still produces a sensible estimate, because nearby states
produce nearby outputs. That's the whole reason "Deep" appears in Deep Q-Network.

### 📉 Turning RL into regression

Here's the trick that makes the whole thing trainable with ordinary Keras. I treat the right-hand
side of the Bellman equation as a **target label**, and fit the network to it with mean squared error:

$$y = r + \gamma \max_{a'} Q_\theta(s', a') \quad \text{(or just } y = r \text{ if the episode ended)}$$

$$\mathcal{L}(\theta) = \big(Q_\theta(s,a) - y\big)^2$$

Now it's just supervised regression — except the labels are generated by the network's own
predictions, and they move as training progresses. This self-referential quality is the main source
of DQN instability, and it's worth keeping in mind when training looks erratic.

### 🎯 Epsilon-greedy exploration

If the agent always takes the action its network currently rates highest, it locks onto whatever it
believed early and never discovers better options. So with probability $\epsilon$ it acts randomly
instead:

$$a = \begin{cases}
\text{random action} & \text{with probability } \epsilon \\[4pt]
\arg\max_a Q_\theta(s,a) & \text{with probability } 1 - \epsilon
\end{cases}$$

$\epsilon$ starts at $1.0$ (pure exploration) and decays multiplicatively toward a floor:

$$\epsilon \leftarrow \max(\epsilon_{\min},\; \epsilon \cdot \lambda), \qquad \lambda = 0.995$$

This is the classic explore/exploit tradeoff — the same tension as a spectrum scan sweeping for a
better channel versus staying on the one that currently works. Sweep too much and throughput
suffers; never sweep and you sit on a degraded channel forever.

### 📥 Experience replay

The naive approach — train on each transition as it happens — breaks gradient descent, because
consecutive timesteps are almost identical and heavily correlated. SGD assumes roughly i.i.d.
samples; feeding it a smooth trajectory violates that badly.

The fix is a **replay buffer**: store every transition $(s, a, r, s', \text{done})$ in a fixed-size
FIFO queue, then train on *random minibatches* sampled from it. This decorrelates consecutive samples
and lets each experience be reused many times.

Think of it as an interleaver in a communication system: burst errors get spread out across the
stream so the decoder sees something closer to independent noise instead of one catastrophic clump.

| Concept | Symbol | Role |
|---|---|---|
| Discount factor | $\gamma$ | How much future reward counts (0.95) |
| Exploration rate | $\epsilon$ | Probability of a random action (1.0 → 0.01) |
| Decay rate | $\lambda$ | Multiplicative decay per replay call (0.995) |
| Replay capacity | — | Transitions retained in memory (2000) |
| Minibatch size | — | Transitions sampled per training step (32) |

## Part 1 — 🌍 Setting Up the Environment

`gymnasium` is the standard toolkit for reinforcement learning environments — it gives every
environment the same `reset()` / `step(action)` interface, so agent code is portable across problems.

I install it along with a pinned TensorFlow version so the results are reproducible.

In [ ]:
!pip install gymnasium

In [ ]:
!pip install tensorflow==2.16.2

Now I create the environment and fix the random seeds. Seeding matters more than usual in RL:
the agent's own randomness (epsilon-greedy, replay sampling) *and* the environment's randomness both
feed into the result, so without seeds two identical runs can look completely different.

In [ ]:
import gymnasium as gym
import numpy as np

# Create the environment
env = gym.make('CartPole-v1')

# Set random seed for reproducibility
np.random.seed(42)
env.reset(seed=42)

📝 **What just happened:**

- `CartPole-v1` — a pole balanced on a cart; the goal is to stop it from falling over.
- `env.reset(seed=42)` returns a tuple `(observation, info)` in modern Gymnasium. Older versions
  returned just the observation, which is why the code below defensively handles both shapes.
- Seeding both NumPy and the environment makes a run repeatable end-to-end.

## Part 2 — 🏗️ Defining the Q-Network

The network maps a 4-dimensional state to 2 Q-values, one per action:

$$\mathbb{R}^4 \;\longrightarrow\; \text{Dense}(24, \text{relu}) \;\longrightarrow\; \text{Dense}(24, \text{relu}) \;\longrightarrow\; \mathbb{R}^2$$

Two design choices deserve attention, because both differ from a typical classifier:

- **Linear output activation, not softmax.** Q-values are unbounded real numbers estimating expected
  return — they are not a probability distribution and must not be squashed into one.
- **MSE loss, not cross-entropy.** This is regression onto a continuous target, not classification.

In [ ]:
# Suppress warnings for a cleaner notebook or console experience
import warnings
warnings.filterwarnings('ignore')

# Disable warnings for a cleaner notebook or console experience
def warn(*args, **kwargs):
    pass
warnings.warn = warn

# Import necessary libraries
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam


def build_model(state_size, action_size):
    model = Sequential()
    model.add(Dense(24, input_dim=state_size, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

state_size = env.observation_space.shape[0]
action_size = env.action_space.n
model = build_model(state_size, action_size)

📝 **Layer by layer:**

| Element | Choice | Reason |
|---|---|---|
| `Sequential` | Linear stack of layers | The architecture has no branching |
| `Dense(24, relu)` ×2 | Fully connected hidden layers | 4 inputs is a tiny state space — 24 units is plenty |
| `input_dim=state_size` | 4 | Matches `env.observation_space.shape[0]` |
| `Dense(action_size, linear)` | 2 outputs | One Q-value per action, unbounded |
| `loss='mse'` | Mean squared error | Regression onto the Bellman target |
| `Adam(lr=0.001)` | Adaptive optimizer | Per-parameter learning rates, robust default |

⚠️ **Note on `input_dim`:** modern Keras prefers an explicit `Input(shape=(state_size,))` layer as the
first entry instead of passing `input_dim` to the first `Dense`. Both work; `input_dim` is what the
original code used and I've kept it so the behaviour is identical.

The network is deliberately small. With a 4-number state and 2 actions, this is a low-dimensional
regression problem — the difficulty lives in the *training signal*, not in the model capacity.

## Part 3 — 📥 Implementing the Replay Buffer

A `deque` with `maxlen=2000` gives me a FIFO ring buffer for free: once it's full, appending a new
transition automatically drops the oldest one. No manual index bookkeeping.

Each stored transition is the complete tuple needed to compute a Bellman target later:

$$(s,\; a,\; r,\; s',\; \text{done})$$

In [ ]:
from collections import deque
import random

memory = deque(maxlen=2000)
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

📝 **Why `done` has to be stored:**

The Bellman target branches on it. If the episode ended, there is no next state to bootstrap from,
so the target is just the reward:

$$y = \begin{cases}
r & \text{if done} \\[4pt]
r + \gamma \max_{a'} Q_\theta(s', a') & \text{otherwise}
\end{cases}$$

Forgetting this flag is a classic DQN bug — the agent starts assigning future value to terminal
states, which by definition have none.

## Part 4 — 🎯 Implementing the Epsilon-Greedy Policy

The exploration schedule in numbers:

| Parameter | Value | Meaning |
|---|---|---|
| `epsilon` | 1.0 | Start fully random — the network knows nothing yet |
| `epsilon_min` | 0.01 | Never stop exploring entirely (1% of actions stay random) |
| `epsilon_decay` | 0.995 | Multiplicative decay applied after each replay call |

In [ ]:
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995

def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model.predict(state)
    return np.argmax(q_values[0])

📝 **Reading `act()`:**

1. Draw a uniform random number. If it's below $\epsilon$, return a random action — **explore**.
2. Otherwise ask the network for $Q(s, \cdot)$ and return `argmax` — **exploit**.

`q_values[0]` indexes into the batch dimension: the network was given a batch of one state, so it
returns shape `(1, 2)` and I want the single row inside it.

Note that a floor of `epsilon_min = 0.01` means exploration never fully stops. That's deliberate —
if the environment shifts, a permanently frozen policy can't notice.

## Part 5 — 🔄 Implementing the Q-Learning Update

This is the core of the algorithm. For each transition in a random minibatch:

1. Compute the Bellman target $y = r + \gamma \max_{a'} Q_\theta(s', a')$, or $y = r$ if terminal.
2. Get the network's current predictions $Q_\theta(s, \cdot)$ for all actions.
3. **Overwrite only the entry for the action actually taken** with $y$.
4. Fit for one step on this modified target vector.

Step 3 is subtle and worth pausing on. I only have evidence about the action I took — I learned
nothing about the action I *didn't* take. By leaving that output equal to the network's own current
prediction, its squared error is exactly zero and it contributes no gradient. The update is surgical:
only the taken action's Q-value moves.

In [ ]:
def replay(batch_size):
    global epsilon
    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        target = reward
        if not done:
            target = reward + gamma * np.amax(model.predict(next_state)[0])
        target_f = model.predict(state)
        target_f[0][action] = target
        model.fit(state, target_f, epochs=1, verbose=0)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

📝 **Line by line:**

| Line | What it does |
|---|---|
| `random.sample(memory, batch_size)` | Draws 32 transitions uniformly at random — this is what breaks temporal correlation |
| `target = reward` | Terminal case: no future to bootstrap from |
| `reward + gamma * np.amax(...)` | Non-terminal case: the Bellman backup |
| `target_f = model.predict(state)` | Current predictions for both actions |
| `target_f[0][action] = target` | Surgical overwrite — only the taken action changes |
| `model.fit(state, target_f, epochs=1)` | One gradient step toward the target |
| `epsilon *= epsilon_decay` | Shift gradually from exploring to exploiting |

⚠️ **Two known limitations of this implementation, worth naming explicitly:**

1. **No target network.** The same network produces both the prediction *and* the target. The target
   therefore moves every time the weights update — like tuning a filter against a reference that
   drifts as you tune it. The 2015 DeepMind DQN paper fixes this with a second, frozen copy of the
   network refreshed every $N$ steps.
2. **Per-sample `predict`/`fit` calls.** Looping over 32 transitions with individual calls is slow;
   a vectorized version would batch all 32 predictions into one call. Correct, but not fast.

⚠️ **Ordering note:** `replay()` references the global `gamma`, which is defined in the training cell
below. Python resolves globals at call time rather than definition time, so this is fine — but the
training cell must be run before `replay()` is ever invoked.

## Part 6 — ⚙️ Training the DQN

Now the pieces come together. For each of 50 episodes:

1. Reset the environment to a fresh starting state.
2. Loop up to 200 timesteps: choose an action, step the environment, store the transition.
3. When the episode terminates, print the score.
4. After the episode, run one `replay()` pass to train on a random minibatch.

The score printed per episode is the number of timesteps the pole stayed up — this is the metric
that should trend upward as learning progresses.

In [ ]:
# Training loop
episodes = 50  # More episodes to ensure sufficient training
batch_size = 32  # Mini-batch size for replay training
gamma = 0.95  # Discount factor for future rewards

for e in range(episodes):
    state = env.reset()
    if isinstance(state, tuple):  # Handle tuple output
        state = state[0]
    state = np.reshape(state, [1, state_size])

    for time in range(200):  # Max steps per episode
        # Choose action using epsilon-greedy policy
        action = act(state)

        # Perform action in the environment
        result = env.step(action)
        if len(result) == 4:  # Handle 4-value output
            next_state, reward, done, _ = result
        else:  # Handle 5-value output
            next_state, reward, done, _, _ = result

        if isinstance(next_state, tuple):  # Handle tuple next_state
            next_state = next_state[0]
        next_state = np.reshape(next_state, [1, state_size])

        # Store experience in memory
        remember(state, action, reward, next_state, done)

        # Update state
        state = next_state

        if done:  # If episode ends
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2}")
            break

    # Train the model using replay memory
    if len(memory) > batch_size:
        replay(batch_size)

env.close()

📝 **What to watch in the output:**

- **Score** should trend upward — noisily. RL learning curves are far rougher than supervised ones,
  because the training data distribution shifts as the policy changes.
- **Epsilon** should fall steadily from 1.0. Early episodes are essentially random flailing; that's
  expected and necessary.

🔍 **On the API-compatibility branches.** The `isinstance(state, tuple)` and `len(result) == 4` checks
exist because Gymnasium changed its API: `reset()` now returns `(obs, info)` and `step()` returns
5 values (`obs, reward, terminated, truncated, info`) instead of 4. This defensive code runs on
either version.

⚠️ **A real limitation here:** the code collapses `terminated` and `truncated` into a single `done`
by taking the third value. Those are semantically different — `terminated` means the pole actually
fell, `truncated` means the time limit was hit while the pole was *still up*. Treating a truncation
as a terminal state tells the agent there was no future value, when in fact things were going well.
On a 200-step cap this is a mild bias, but it's a genuine correctness wrinkle rather than a stylistic one.

🐌 **Expect this to be slow.** Every `replay()` call makes 96 separate `predict`/`fit` calls
(32 transitions × 3 calls each), and each carries full Keras overhead. 50 episodes takes a while.

## Part 7 — 📊 Evaluating the Performance

Evaluation differs from training in one critical way: **exploration is switched off**. The agent
acts purely greedily, always taking `argmax` over Q-values. No epsilon, no random actions, no
learning — this measures the policy as it actually stands.

I run 10 episodes and track total reward per episode, which on CartPole equals the number of
timesteps survived.

In [ ]:
# Evaluation loop
evaluation_episodes = 10  # Number of evaluation episodes
scores = []  # Track scores for performance metrics

for e in range(evaluation_episodes):
    state = env.reset()
    if isinstance(state, tuple):  # Handle tuple output
        state = state[0]
    state = np.reshape(state, [1, state_size])

    total_reward = 0  # Track total reward per episode

    for time in range(200):  # Max steps per episode
        # Choose the greedy action
        action = np.argmax(model.predict(state)[0])

        # Perform action in the environment
        result = env.step(action)
        if len(result) == 4:  # Handle 4-value output
            next_state, reward, done, _ = result
        else:  # Handle 5-value output
            next_state, reward, done, _, _ = result

        if isinstance(next_state, tuple):  # Handle tuple next_state
            next_state = next_state[0]
        next_state = np.reshape(next_state, [1, state_size])

        state = next_state
        total_reward += reward

        if done:  # If episode ends
            print(f"Evaluation Episode: {e+1}/{evaluation_episodes}, Score: {time}, Total Reward: {total_reward}")
            scores.append(total_reward)
            break

# Summary of evaluation performance
print(f"Average Reward: {np.mean(scores):.2f}, Max Reward: {np.max(scores)}, Min Reward: {np.min(scores)}")

env.close()

📝 **Reading the evaluation numbers:**

| Metric | What it tells me |
|---|---|
| **Average reward** | Overall policy quality across episodes |
| **Max reward** | Best case — what the agent achieves when conditions favour it |
| **Min reward** | Worst case — how badly it fails on an unlucky start |

A **large gap between max and min** is the interesting signal. It means the policy is brittle: it
handles some initial conditions well and collapses on others. That's the RL equivalent of a
controller that's stable in the nominal operating region but not across the full envelope.

For reference, CartPole-v1 is conventionally considered "solved" at an average of **195+** over
100 consecutive episodes. With only 50 training episodes and one replay pass each, I should expect
to land well short of that — this notebook is about understanding the mechanism, not about
maximizing the score.

⚠️ **Note:** `scores.append()` sits inside the `if done:` branch, so an episode that survives the full
200 steps without terminating never records a score. On a well-trained agent that would silently
skew the statistics upward — another thing to fix in a production version.

## 🎯 Practice 1 — Reward Shaping to Encourage Longer Episodes

**Objective:** modify the reward so the agent is actively discouraged from letting the pole drift
far from vertical, rather than only being told after the fact that it fell.

**The problem with the default reward.** CartPole gives $+1$ per timestep survived — nothing more.
It carries no information about *how close to failing* the agent is. A pole at 11.9° and a pole at
0.1° earn exactly the same reward, right up until the moment one of them ends the episode. The
learning signal is sparse and late.

**Reward shaping** adds a denser, more informative signal:

$$r' = r - \mathbb{1}\big[\,|\theta| > 0.1\,\big]$$

The agent now gets immediate feedback that a large angle is bad, instead of discovering it only on
termination. This is the same instinct as a proportional control term: penalize the error
continuously instead of waiting for the system to hit a hard limit.

In [ ]:
# Function to modify the reward to encourage longer episodes
def modify_reward(reward, next_state):
    # Extract the pole angle (index 2 of the state vector) and penalize large deviations
    pole_angle = abs(next_state[0][2])
    penalty = 1 if pole_angle > 0.1 else 0  # Apply penalty if angle is large
    return reward - penalty  # Adjust reward


# Re-initialize so this experiment starts from a clean slate
model = build_model(state_size, action_size)
memory = deque(maxlen=2000)
epsilon = 1.0

episodes = 30
batch_size = 32
gamma = 0.95

for e in range(episodes):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    state = np.reshape(state, [1, state_size])

    for time in range(200):
        action = act(state)

        result = env.step(action)
        if len(result) == 4:
            next_state, reward, done, _ = result
        else:
            next_state, reward, done, _, _ = result

        if isinstance(next_state, tuple):
            next_state = next_state[0]
        next_state = np.reshape(next_state, [1, state_size])

        # Apply the shaped reward instead of the raw environment reward
        shaped_reward = modify_reward(reward, next_state)

        remember(state, action, shaped_reward, next_state, done)
        state = next_state

        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.2}")
            break

    if len(memory) > batch_size:
        replay(batch_size)

env.close()

📝 **What changed and why it matters:**

`next_state` has shape `(1, 4)` after the reshape, so the pole angle is at `next_state[0][2]` —
indexing `next_state[2]` would silently read the wrong thing (or crash), which is an easy mistake here.

⚠️ **Reward shaping is a genuinely dangerous tool.** By changing the reward I change what "optimal"
means — the agent will faithfully optimize whatever I actually wrote, not what I intended. If the
shaping term is badly designed, the agent finds a way to farm the shaped reward while ignoring the
real goal. Here the penalty is aligned with the true objective (keeping the pole up), so it helps.
That alignment is not automatic.

| Reward design | Signal density | Risk |
|---|---|---|
| `+1` per timestep (default) | Sparse, late | Slow learning — feedback only at failure |
| `+1` − angle penalty (shaped) | Dense, immediate | Must stay aligned with the true goal |

## 🧪 Practice 2 — Early Stopping Based on Episode Length

**Objective:** stop training automatically once the agent consistently reaches a target episode
length, instead of burning compute on a policy that has already converged.

The CartPole convention: **195 steps sustained over 100 consecutive episodes** counts as solved.
Requiring a *streak* rather than a single good episode is the whole point — one lucky episode proves
nothing, since initial conditions vary. This is a debouncing condition: don't act on a single
threshold crossing, act on a sustained one.

In [ ]:
# Early stopping parameters
consecutive_success_threshold = 100  # Number of consecutive episodes required
success_episode_length = 195         # Steps needed for an episode to count as a success
episode_lengths = []                 # Track the length of every episode

# Re-initialize for a clean run
model = build_model(state_size, action_size)
memory = deque(maxlen=2000)
epsilon = 1.0

episodes = 300
batch_size = 32
gamma = 0.95

for e in range(episodes):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    state = np.reshape(state, [1, state_size])

    episode_length = 0

    for time in range(200):
        action = act(state)

        result = env.step(action)
        if len(result) == 4:
            next_state, reward, done, _ = result
        else:
            next_state, reward, done, _, _ = result

        if isinstance(next_state, tuple):
            next_state = next_state[0]
        next_state = np.reshape(next_state, [1, state_size])

        remember(state, action, reward, next_state, done)
        state = next_state
        episode_length = time + 1

        if done:
            break

    # Record the length of this episode, whether or not it terminated early
    episode_lengths.append(episode_length)
    print(f"Episode: {e+1}/{episodes}, Length: {episode_length}, Epsilon: {epsilon:.2}")

    if len(memory) > batch_size:
        replay(batch_size)

    # Early stopping check: a sustained streak, not a single good episode
    if len(episode_lengths) >= consecutive_success_threshold and all(
        length >= success_episode_length
        for length in episode_lengths[-consecutive_success_threshold:]
    ):
        print(f"Early stopping at episode {e+1}: agent consistently reaches max episode length. ✅")
        break

env.close()

📝 **Design notes:**

- `episode_length = time + 1` because `time` is a 0-based loop index — off by one otherwise.
- The append happens **outside** the inner loop, so episodes that run the full 200 steps without
  terminating still get recorded. This is exactly the bug I flagged in the evaluation loop above,
  fixed here.
- `all(...)` over the last 100 entries is the streak check. The guard
  `len(episode_lengths) >= consecutive_success_threshold` prevents it firing prematurely, since
  `all()` over a short list would be trivially satisfiable.

⚠️ **This won't actually trigger here.** With a 200-step cap and one replay pass per episode, the
agent will not reach a 100-episode streak at 195+ within 300 episodes. The mechanism is correct;
the training budget isn't sufficient to exercise it. To see it fire I'd need a target network,
vectorized replay, and several thousand episodes.

## ⚙️ Practice 3 — Hybrid Epsilon Decay Schedule

**Objective:** start with **linear** epsilon decay, then switch to **exponential** decay after a
set number of episodes.

The two schedules have opposite characters:

$$\epsilon_{\text{linear}} \leftarrow \max(\epsilon - 0.01,\; 0.01)$$

$$\epsilon_{\text{exponential}} \leftarrow \max(\epsilon \cdot 0.99,\; 0.01)$$

| Schedule | Early behaviour | Late behaviour |
|---|---|---|
| **Linear** | Drops steadily, constant absolute step | Reaches the floor abruptly |
| **Exponential** | Drops fast when $\epsilon$ is large | Long slow tail near the floor |
| **Hybrid** | Linear: decisive early reduction | Exponential: gentle final approach |

The hybrid gets a decisive early cut in randomness followed by a soft landing — structurally the
same idea as a learning-rate schedule that steps down early then anneals smoothly.

In [ ]:
def decay_epsilon(epsilon, episode, switch_episode=100):
    if episode < switch_episode:
        return max(epsilon - 0.01, 0.01)   # Linear decay
    else:
        return max(epsilon * 0.99, 0.01)   # Exponential decay


# Compare the three schedules over 300 episodes without training anything
eps_linear, eps_exponential, eps_hybrid = 1.0, 1.0, 1.0
history = {'linear': [], 'exponential': [], 'hybrid': []}

for episode in range(300):
    eps_linear = max(eps_linear - 0.01, 0.01)
    eps_exponential = max(eps_exponential * 0.99, 0.01)
    eps_hybrid = decay_epsilon(eps_hybrid, episode, switch_episode=100)

    history['linear'].append(eps_linear)
    history['exponential'].append(eps_exponential)
    history['hybrid'].append(eps_hybrid)

for ep in [0, 50, 99, 100, 150, 200, 299]:
    print(f"Episode {ep:>3} | linear={history['linear'][ep]:.4f} "
          f"| exponential={history['exponential'][ep]:.4f} "
          f"| hybrid={history['hybrid'][ep]:.4f}")

Now I plot the three schedules so the difference in shape is visible rather than inferred from
numbers.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(history['linear'], label='Linear decay', linewidth=2)
plt.plot(history['exponential'], label='Exponential decay', linewidth=2)
plt.plot(history['hybrid'], label='Hybrid (linear -> exponential @100)', linewidth=2, linestyle='--')
plt.axvline(100, color='gray', linestyle=':', label='Switch point')
plt.xlabel('Episode')
plt.ylabel('Epsilon (exploration rate)')
plt.title('Comparing Epsilon Decay Schedules')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Finally, I run a short training loop that actually uses the hybrid schedule. Note that `replay()`
applies its own multiplicative decay, so I override `epsilon` explicitly after each episode to keep
the hybrid schedule in control.

In [ ]:
# Re-initialize for a clean run
model = build_model(state_size, action_size)
memory = deque(maxlen=2000)
epsilon = 1.0

episodes = 30
batch_size = 32
gamma = 0.95

for e in range(episodes):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    state = np.reshape(state, [1, state_size])

    for time in range(200):
        action = act(state)

        result = env.step(action)
        if len(result) == 4:
            next_state, reward, done, _ = result
        else:
            next_state, reward, done, _, _ = result

        if isinstance(next_state, tuple):
            next_state = next_state[0]
        next_state = np.reshape(next_state, [1, state_size])

        remember(state, action, reward, next_state, done)
        state = next_state

        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

    if len(memory) > batch_size:
        replay(batch_size)

    # Override replay()'s internal decay with the hybrid schedule
    epsilon = decay_epsilon(epsilon, e, switch_episode=100)

env.close()

📝 **Why the explicit override matters:** `replay()` contains `epsilon *= epsilon_decay`, which would
otherwise compete with `decay_epsilon()`. Two decay mechanisms operating on the same variable is a
subtle bug — the effective schedule becomes the product of both, not the one I designed. Setting
`epsilon` explicitly after the replay call makes the hybrid schedule authoritative.

## 📊 Summary

I built a complete Deep Q-Network from primitives and used it to learn a control policy for CartPole.

### Components

| Component | Implementation | Purpose |
|---|---|---|
| 🌍 **Environment** | Gymnasium `CartPole-v1` | 4-D continuous state, 2 discrete actions |
| 🏗️ **Q-network** | `Dense(24) → Dense(24) → Dense(2, linear)` | Approximates $Q(s,a)$ where a table can't |
| 📥 **Replay buffer** | `deque(maxlen=2000)` | Decorrelates samples, enables experience reuse |
| 🎯 **Policy** | Epsilon-greedy, $1.0 \to 0.01$ | Balances exploration against exploitation |
| 🔄 **Update rule** | Bellman target + MSE | Reframes RL as supervised regression |
| ⚙️ **Training** | 50 episodes × 200 steps | Agent generates its own training data |
| 📊 **Evaluation** | 10 greedy episodes | Measures the policy with exploration off |

### Hyperparameters

| Parameter | Value | Effect if raised |
|---|---|---|
| $\gamma$ (discount) | 0.95 | More far-sighted, but higher-variance targets |
| $\epsilon$ decay | 0.995 | Slower shift from exploring to exploiting |
| Replay capacity | 2000 | More diverse batches, more stale experiences |
| Batch size | 32 | Smoother gradients, slower per step |
| Learning rate | 0.001 | Faster movement, higher divergence risk |

### Key equations

$$Q(s, a) = r + \gamma \max_{a'} Q(s', a') \qquad \text{(Bellman)}$$

$$\mathcal{L}(\theta) = \big(Q_\theta(s,a) - y\big)^2 \qquad \text{(regression loss)}$$

$$\epsilon \leftarrow \max(\epsilon_{\min},\; \epsilon \cdot \lambda) \qquad \text{(exploration decay)}$$

### 🎓 What I take away

1. **RL is regression with self-generated, moving labels.** Once I saw the Bellman target as a label,
   the whole thing reduced to `model.fit()` — the novelty is in *where the label comes from*, not in
   the training machinery.
2. **The interesting engineering is in the data pipeline, not the model.** A 2-layer MLP is trivial.
   The replay buffer, the exploration schedule, and the reward design are what determine whether it
   learns at all.
3. **Exploration is a scheduled resource.** Too little and the agent commits early to a bad policy;
   too much and it never exploits what it learned. Same tradeoff as a spectrum scan.
4. **Reward shaping changes the objective.** The agent optimizes the reward I write, not the goal I
   have in mind. Alignment between the two is my responsibility.
5. **This implementation is honest but not production-grade.** No target network, per-sample
   `predict`/`fit`, and `terminated`/`truncated` collapsed into one flag. Each is a real, named gap
   rather than a hidden one.

### ⚠️ Gaps to close next

| Gap | Consequence | Fix |
|---|---|---|
| No target network | Targets drift as weights update → unstable | Frozen copy of the network, synced every $N$ steps |
| Per-sample `predict`/`fit` | Extremely slow training | Vectorize the whole minibatch into one call |
| `terminated` vs `truncated` merged | Truncation wrongly treated as terminal | Unpack both and use `terminated` for the Bellman branch |
| Score only recorded on `done` | Full-length episodes silently dropped | Record outside the inner loop (as in Practice 2) |

## 🧪 Sandbox

Space to experiment. Suggestions, roughly in order of value:

**1. Add a target network** — the single highest-impact fix. Keep a frozen copy of the model, compute
Bellman targets from *it*, and sync it to the live weights every $N$ steps:

```python
target_model = build_model(state_size, action_size)
target_model.set_weights(model.get_weights())

# in replay(): use target_model.predict(next_state) for the max term
# every N steps: target_model.set_weights(model.get_weights())
```

**2. Vectorize `replay()`** — batch the 32 transitions into two `predict` calls and one `fit` call
instead of 96 separate calls. Expect a large speedup:

```python
states = np.vstack([t[0] for t in minibatch])
next_states = np.vstack([t[3] for t in minibatch])
q_current = model.predict(states, verbose=0)
q_next = model.predict(next_states, verbose=0)
# build targets from q_next, then one model.fit(states, q_current, ...)
```

**3. Handle `terminated` and `truncated` separately** — unpack all 5 return values and use only
`terminated` for the Bellman branch, so hitting the time limit isn't misread as failure.

**4. Sweep $\gamma$** — try 0.9, 0.95, 0.99. How far ahead does the agent need to plan on a problem
where failure is always ~20 steps away?

**5. Change the architecture** — try `Dense(64) → Dense(64)`, or a single hidden layer. Does more
capacity help on a 4-dimensional state, or is the bottleneck entirely in the training signal?

**6. Plot the learning curve** — collect episode scores and plot them with a rolling mean. RL curves
are noisy enough that the raw trace is hard to read without smoothing.

**7. Try a different environment** — `MountainCar-v0` or `Acrobot-v1` use the same interface. Both
have sparser rewards, which makes exploration much harder and shows why reward shaping matters.

In [ ]:
# 🧪 Sandbox — experiment freely